In [12]:
from pathlib import Path
import pandas as pd

In [13]:
RAW = Path("../01_data/raw")
CLEAN = Path("../01_data/interim/cleaned")
CLEAN.mkdir(parents=True, exist_ok=True)

ben_file = RAW / "ben_sarc" / "Ben-Sarc.xlsx"
banglasarc_file = RAW / "banglasarc" / "SarcasDetection.csv"
banglasarc3_file = RAW / "banglasarc3" / "BanglaSarc3.xlsx"

In [14]:
def load_table(file_path):
    suffix = file_path.suffix.lower()

    if suffix == ".csv":
        for enc in ["utf-8", "utf-8-sig", "latin1", "cp1252"]:
            try:
                return pd.read_csv(file_path, encoding=enc)
            except Exception:
                pass
        raise ValueError(f"Could not read CSV file: {file_path}")
    elif suffix in [".xlsx", ".xls"]:
        return pd.read_excel(file_path)
    else:
        raise ValueError(f"Unsupported file type: {suffix}")

In [15]:
ben_df = load_table(ben_file)
banglasarc_df = load_table(banglasarc_file)
banglasarc3_df = load_table(banglasarc3_file)

In [16]:
def standardize_dataset(df, text_col, label_col, dataset_name):
    out = df[[text_col, label_col]].copy()
    out = out.rename(columns={
        text_col: "text",
        label_col: "label_original"
    })

    out["dataset_name"] = dataset_name

    # remove missing text
    out = out.dropna(subset=["text"]).copy()

    # strip whitespace from text if text is string
    out["text"] = out["text"].astype(str).str.strip()

    # remove fully empty strings
    out = out[out["text"] != ""].copy()

    return out

In [17]:
ben_clean = standardize_dataset(
    ben_df,
    text_col="Text",
    label_col="Polarity",
    dataset_name="ben_sarc"
)

banglasarc_clean = standardize_dataset(
    banglasarc_df,
    text_col="Comments",
    label_col="Label",
    dataset_name="banglasarc"
)

banglasarc3_clean = standardize_dataset(
    banglasarc3_df,
    text_col="Bangla Comments",
    label_col="Sarcasm Label",
    dataset_name="banglasarc3"
)

In [18]:
# Ben-Sarc: already binary
ben_clean["label"] = ben_clean["label_original"].astype(int)

# BanglaSarc: likely float 0.0 / 1.0
banglasarc_clean["label"] = banglasarc_clean["label_original"].astype(int)

# BanglaSarc3: keep ternary labels as text for now
banglasarc3_clean["label"] = banglasarc3_clean["label_original"].astype(str).str.strip()

In [19]:
final_cols = ["text", "label", "label_original", "dataset_name"]

ben_clean = ben_clean[final_cols]
banglasarc_clean = banglasarc_clean[final_cols]
banglasarc3_clean = banglasarc3_clean[final_cols]


In [20]:
print(ben_clean.shape)
display(ben_clean.head())

print(banglasarc_clean.shape)
display(banglasarc_clean.head())

print(banglasarc3_clean.shape)
display(banglasarc3_clean.head())

(25636, 4)


,text,label,label_original,dataset_name
0,শুধু মাত্র এই পোস্টে কমেন্ট করার জন্য বাড়ির এ...,1,1,ben_sarc
1,সাথে আছে বুক ভরা চুল ।,1,1,ben_sarc
2,ভাই মিথ্যা কথা বইলেন না আপনি ভিপিএন ইউজ করে পো...,1,1,ben_sarc
3,দিঠি আর ডিংকার তো ইদুরের মত দাঁত আছে । ওরা জাল...,1,1,ben_sarc
4,ওদেরকে পানির জাহাজে নেওয়া হোক । তারা বিশেষ বিম...,1,1,ben_sarc


(5112, 4)


,text,label,label_original,dataset_name
0,আমি মনে করি যখন মেয়েরা উদ্দেশ্য অনুসারে বোবা ...,1,1.0,banglasarc
1,আমি যখন কোনও গুরুত্বপূর্ণ প্রশ্ন জিজ্ঞাসা করি ...,1,1.0,banglasarc
2,বাহ ... আমি দেখতে দেখতে পাচ্ছি সত্যিই দোষী দোষ...,1,1.0,banglasarc
3,"হাহ, মজার বিষয়।#আনোয়েড #গ্রুআপ",1,1.0,banglasarc
4,থান্ডার আমাকে 830 এ জেগে উঠলে এটি ভালবাসুন,1,1.0,banglasarc


(12072, 4)


,text,label,label_original,dataset_name
0,মুরগির ঝোল আর ভাত পছন্দ হলে সেটাই খাবেন,Neutral,Neutral,banglasarc3
1,টং দোকানের চা ভালো লাগলে সেখানে যান,Neutral,Neutral,banglasarc3
2,নিজের পছন্দের জিনিস উপভোগ করুন,Neutral,Neutral,banglasarc3
3,অপরাধ দমনে সমাজের সকল স্তরের মানুষকে এগিয়ে আস...,Neutral,Neutral,banglasarc3
4,শিক্ষা ও নৈতিকতার চর্চা শিশুদের রক্ষায় গুরুত্...,Neutral,Neutral,banglasarc3


In [21]:
print("Ben-Sarc labels:")
print(ben_clean["label"].value_counts(dropna=False))

print("\nBanglaSarc labels:")
print(banglasarc_clean["label"].value_counts(dropna=False))

print("\nBanglaSarc3 labels:")
print(banglasarc3_clean["label"].value_counts(dropna=False))

Ben-Sarc labels:
label
1    12818
0    12818
Name: count, dtype: int64

BanglaSarc labels:
label
0    3159
1    1953
Name: count, dtype: int64

BanglaSarc3 labels:
label
Neutral          4055
Non-Sarcastic    4009
Sarcastic        4008
Name: count, dtype: int64


In [22]:
ben_clean.to_csv(CLEAN / "ben_sarc_clean.csv", index=False)
banglasarc_clean.to_csv(CLEAN / "banglasarc_clean.csv", index=False)
banglasarc3_clean.to_csv(CLEAN / "banglasarc3_clean.csv", index=False)

print("Saved cleaned files to:", CLEAN)

Saved cleaned files to: ../01_data/interim/cleaned
